# Day 6 — Three-Asset Brownian-Bridge Conditioning (M2/M3)

**Outcome.** This notebook implements the observation-endpoint M2/M3 estimators for
the frozen RC-L and RC-A contracts and cross-checks them against direct M0/M1.

For every live observation interval it stores the exact marginal exits (g_i),
the exact bivariate co-exits (h_{ij}), a fresh nested conditional-bridge estimate
of (q_3), and the inclusion–exclusion survival probability

\[
p_3=1-\sum_i g_i+\sum_{i<j}h_{ij}-q_3.
\]

The Day 4 logistic is **not** reused: its OOS gate failed and its trained signature
does not cover path-specific effective barriers. The fresh nested bridge bank is
therefore the explicit fallback. Discrete coupons/autocalls remain endpoint events;
continuous KI is integrated out only through segment survival weights.

## Method and chart contract

- **M0:** direct fine-grid MC; **M1:** direct fine-grid scrambled-Sobol RQMC.
- **M2:** endpoint MC + conditional BB weight; **M3:** endpoint scrambled-Sobol RQMC + conditional BB weight.
- Lee Corollary 5.2 transforms lower-barrier survival on an interval to the upper-barrier
  inputs (x^*=X(t_0)-X(t_1)), (b^*=X(t_0)-\log B), (T^*=t_1-t_0).
- The nested (q_3) estimate is anchored to exact (h_{ij}), projected to its feasible
  Fréchet/inclusion–exclusion interval, and both raw and projected values are saved.
- Static notebook charts answer two questions: (1) do M0–M3 agree within uncertainty,
  and (2) how do interval (p_3) and cumulative weights evolve? Price uses a categorical
  dot-and-interval comparison; probabilities use observation-index lines. Explicit
  blue/gold/orange/olive colors plus marker/line-style differences avoid color-only encoding.

In [ ]:
from pathlib import Path
import json
import math
import subprocess
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import openpyxl
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "config" / "core_project_config.json").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "config" / "core_project_config.json").is_file():
    raise FileNotFoundError("Start Jupyter from inside the autocallable-rqmc repository")
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from autocallable_direct import (
    equicorrelation,
    make_psd_correlation,
    parse_research_contract,
    simulate_direct_replication,
    stable_seed,
)
from autocallable_bb import (
    BridgeBank,
    generate_bridge_bank,
    generate_endpoint_paths,
    segment_probabilities,
    simulate_conditioned_replication,
    simulate_direct_rqmc_replication,
    summarise_method_replications,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
pd.options.display.float_format = "{:,.8f}".format
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 200,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.titleweight": "semibold",
    "axes.labelsize": 10,
    "axes.edgecolor": "#475569",
    "axes.labelcolor": "#1f2937",
    "text.color": "#1f2937",
    "xtick.color": "#475569",
    "ytick.color": "#475569",
    "grid.color": "#dbe3ec",
    "grid.linewidth": 0.75,
    "grid.alpha": 0.9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})

CONFIG_PATH = PROJECT_ROOT / "config" / "core_project_config.json"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "day6_three_asset_bb_conditioning"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
with CONFIG_PATH.open(encoding="utf-8") as handle:
    CONFIG = json.load(handle)

SEED = 20260806
DIRECT_PATHS = 2048
DIRECT_REPLICATIONS = 4
DIRECT_STEPS_PER_YEAR = 252
CONDITIONED_PATHS = 2048
CONDITIONED_REPLICATIONS = 6
INNER_PATHS = 256
BRIDGE_SUBSTEPS = 16
BESSEL_TERMS = 12
REFERENCE_INNER_PATHS = 4096
REFERENCE_SUBSTEPS = 64
print("Project:", PROJECT_ROOT)
print("Evidence directory:", OUTPUT_DIR)

## 1. Load frozen contracts and market inputs

In [ ]:
def is_number(value):
    return isinstance(value, (int, float, np.integer, np.floating)) and not isinstance(value, bool) and np.isfinite(value)


def read_market_inputs(path):
    workbook = openpyxl.load_workbook(path, data_only=True, read_only=False)
    setup = workbook["Setup"]
    snapshot = workbook["Underlying_Snapshot"]
    history = workbook["Underlying_History"]
    market_history = workbook["Market_History"]
    tickers = [snapshot.cell(row, 2).value for row in (6, 7, 8)]
    dividend_yields = np.array([snapshot.cell(row, 5).value for row in (6, 7, 8)], dtype=float) / 100.0
    volatilities = np.array([snapshot.cell(row, 8).value for row in (6, 7, 8)], dtype=float) / 100.0
    histories = []
    for date_column, value_column, ticker in zip((1, 4, 7), (2, 5, 8), tickers):
        values = {}
        for row in range(6, history.max_row + 1):
            date_value = history.cell(row, date_column).value
            level = history.cell(row, value_column).value
            if hasattr(date_value, "year") and is_number(level) and level > 0:
                values[pd.Timestamp(date_value)] = float(level)
        histories.append(pd.Series(values, name=ticker).sort_index())
    prices = pd.concat(histories, axis=1, join="inner").dropna()
    correlation = make_psd_correlation(np.log(prices / prices.shift(1)).dropna().corr().to_numpy())
    rates = []
    for row in range(6, market_history.max_row + 1):
        date_value = market_history.cell(row, 10).value
        rate_pct = market_history.cell(row, 11).value
        if hasattr(date_value, "year") and is_number(rate_pct) and rate_pct > 0:
            rates.append((pd.Timestamp(date_value), float(rate_pct) / 100.0))
    rates = pd.Series(dict(rates)).sort_index()
    return {
        "tickers": tickers,
        "dividend_yields": dividend_yields,
        "volatilities": volatilities,
        "correlation": correlation,
        "risk_free_rate": float(rates.iloc[-1]),
        "rate_as_of": rates.index[-1],
        "workbook_as_of": pd.Timestamp(setup["B8"].value),
    }


market = read_market_inputs(PROJECT_ROOT / CONFIG["market_data"]["relative_path"])
rc_l = parse_research_contract(CONFIG, "RC-L")
rc_a = parse_research_contract(CONFIG, "RC-A")
parameter_sets = {
    "RC-L": {
        "contract": rc_l,
        "risk_free_rate": 0.03,
        "dividend_yields": np.zeros(3),
        "volatilities": np.full(3, 0.20),
        "correlation": equicorrelation(0.40),
    },
    "RC-A": {
        "contract": rc_a,
        "risk_free_rate": market["risk_free_rate"],
        "dividend_yields": market["dividend_yields"],
        "volatilities": market["volatilities"],
        "correlation": market["correlation"],
    },
}
market_inputs = pd.DataFrame({
    "ticker": market["tickers"],
    "dividend_yield": market["dividend_yields"],
    "volatility": market["volatilities"],
})
display(market_inputs)
display(pd.DataFrame(market["correlation"], index=market["tickers"], columns=market["tickers"]))
print(f"Workbook as of {market['workbook_as_of'].date()}; rate {market['risk_free_rate']:.4%} as of {market['rate_as_of'].date()}")

## 2. Probability-component regression checks

In [ ]:
appendix_x = np.array([[0.01, 0.02, 0.03]])
appendix_b = np.full((1, 3), 0.10)
appendix_sigma = np.full(3, 0.20)
appendix_corr = equicorrelation(0.40)
appendix_bank = generate_bridge_bank(0.50, appendix_sigma, appendix_corr, 4096, 32, stable_seed(SEED, "appendix"))
appendix = segment_probabilities(appendix_x, appendix_b, 0.50, appendix_sigma, appendix_corr, appendix_bank)
appendix_check = pd.DataFrame({
    "component": ["g1", "g2", "g3", "h12", "h13", "h23", "q3", "p3"],
    "value": np.concatenate([appendix["g"][0], appendix["h"][0], appendix["q3"], appendix["p3"]]),
})
display(appendix_check)
assert np.max(np.abs(appendix["h"][0] - np.array([0.2375, 0.2565, 0.2790]))) < 6e-4
assert 0.0 <= appendix["p3"][0] <= 1.0

## 3. M0–M3 full-price experiment

In [ ]:
records = []
segment_frames = []
weight_frames = []
experiment_started = time.perf_counter()

for contract_id, params in parameter_sets.items():
    contract = params["contract"]
    coupon = contract.baseline_annual_coupon
    for replication in range(DIRECT_REPLICATIONS):
        m0 = simulate_direct_replication(
            contract=contract,
            risk_free_rate=params["risk_free_rate"],
            dividend_yields=params["dividend_yields"],
            volatilities=params["volatilities"],
            correlation=params["correlation"],
            annual_coupon=coupon,
            n_paths=DIRECT_PATHS,
            steps_per_year=DIRECT_STEPS_PER_YEAR,
            seed=stable_seed(SEED, contract_id, "M0", replication),
            batch_size=512,
        )
        m0.update({"method": "M0", "replication": replication, "outer_method": "mc-direct"})
        records.append(m0)
        m1 = simulate_direct_rqmc_replication(
            contract=contract,
            risk_free_rate=params["risk_free_rate"],
            dividend_yields=params["dividend_yields"],
            volatilities=params["volatilities"],
            correlation=params["correlation"],
            annual_coupon=coupon,
            n_paths=DIRECT_PATHS,
            steps_per_year=DIRECT_STEPS_PER_YEAR,
            seed=stable_seed(SEED, contract_id, "M1", replication),
        )
        m1["replication"] = replication
        records.append(m1)

    bank_cache = {}
    bridge_bank_seed = stable_seed(SEED, contract_id, "production-bridge-bank")
    for method, outer_method in (("M2", "mc"), ("M3", "rqmc")):
        for replication in range(CONDITIONED_REPLICATIONS):
            result, segment, weights = simulate_conditioned_replication(
                contract=contract,
                risk_free_rate=params["risk_free_rate"],
                dividend_yields=params["dividend_yields"],
                volatilities=params["volatilities"],
                correlation=params["correlation"],
                annual_coupon=coupon,
                n_paths=CONDITIONED_PATHS,
                outer_method=outer_method,
                seed=stable_seed(SEED, contract_id, method, replication),
                bridge_bank_seed=bridge_bank_seed,
                inner_paths=INNER_PATHS,
                bridge_substeps=BRIDGE_SUBSTEPS,
                bessel_terms=BESSEL_TERMS,
                bank_cache=bank_cache,
                paired_bernoulli=True,
                return_diagnostics=True,
            )
            result["replication"] = replication
            records.append(result)
            segment["contract_id"] = contract_id
            segment["method"] = method
            segment["replication"] = replication
            segment_frames.append(segment)
            weights["contract_id"] = contract_id
            weights["method"] = method
            weights["replication"] = replication
            weight_frames.append(weights)

replications = pd.DataFrame(records)
segment_diagnostics = pd.concat(segment_frames, ignore_index=True)
weight_distribution = pd.concat(weight_frames, ignore_index=True)
method_summary = summarise_method_replications(replications)
print(f"Experiment runtime: {time.perf_counter() - experiment_started:.2f} seconds")
display(method_summary[[
    "contract_id", "method", "replications", "n_paths_per_replication",
    "total_value_mean", "total_value_se", "fair_coupon_mean", "fair_coupon_se",
    "continuous_ki_survival_probability_mean", "runtime_seconds_mean",
]])

## 4. Price agreement, component contributions and paired Bernoulli bridge

In [ ]:
component_columns = ["coupon_value", "early_redemption_principal", "surviving_notional", "terminal_ki_loss"]
component_comparison = (
    replications.groupby(["contract_id", "method"], as_index=False)[component_columns]
    .mean()
)

agreement_rows = []
for contract_id in parameter_sets:
    subset = method_summary[method_summary["contract_id"] == contract_id].set_index("method")
    reference = subset.loc["M0"]
    for method, row in subset.iterrows():
        difference = row["total_value_mean"] - reference["total_value_mean"]
        combined_se = math.sqrt(row["total_value_se"] ** 2 + reference["total_value_se"] ** 2)
        tolerance = max(1.25, 3.0 * combined_se)
        agreement_rows.append({
            "contract_id": contract_id,
            "method": method,
            "reference_method": "M0",
            "price_difference": difference,
            "combined_se": combined_se,
            "tolerance": tolerance,
            "pass": abs(difference) <= tolerance,
        })
price_agreement = pd.DataFrame(agreement_rows)

paired_rows = []
conditioned = replications[replications["method"].isin(["M2", "M3"])]
for (contract_id, method), group in conditioned.groupby(["contract_id", "method"]):
    values = group["conditional_minus_paired_bernoulli"].astype(float)
    se = values.std(ddof=1) / math.sqrt(len(values))
    tolerance = max(0.35, 3.0 * se)
    paired_rows.append({
        "contract_id": contract_id,
        "method": method,
        "paired_mean_difference": values.mean(),
        "paired_difference_se": se,
        "tolerance": tolerance,
        "pass": abs(values.mean()) <= tolerance,
    })
paired_comparison = pd.DataFrame(paired_rows)
display(price_agreement)
display(component_comparison)
display(paired_comparison)

## 5. Fresh nested-fallback approximation audit

In [ ]:
params = parameter_sets["RC-A"]
contract = params["contract"]
audit_endpoints = generate_endpoint_paths(
    256,
    contract.observation_times,
    params["risk_free_rate"],
    params["dividend_yields"],
    params["volatilities"],
    params["correlation"],
    method="rqmc",
    seed=stable_seed(SEED, "approx-audit-endpoints"),
)
audit_rows = []
selected_intervals = [0, len(contract.observation_times) // 2, len(contract.observation_times) - 1]
previous_times = np.concatenate([[0.0], np.asarray(contract.observation_times[:-1])])
for interval_index in selected_intervals:
    start = np.zeros((256, 3)) if interval_index == 0 else audit_endpoints[:, interval_index - 1, :]
    end = audit_endpoints[:, interval_index, :]
    dt = contract.observation_times[interval_index] - previous_times[interval_index]
    all_x = start - end
    all_b = np.maximum(start - math.log(contract.ki_barrier_ratio), 0.0)
    eligible = np.flatnonzero(np.all((all_b > 0.0) & (all_x < all_b), axis=1))
    if len(eligible) < 24:
        raise RuntimeError(f"insufficient non-breached audit endpoints in interval {interval_index + 1}")
    sample_index = eligible[np.linspace(0, len(eligible) - 1, 24, dtype=int)]
    x = all_x[sample_index]
    b = all_b[sample_index]
    production_bank = generate_bridge_bank(
        dt, params["volatilities"], params["correlation"], INNER_PATHS, BRIDGE_SUBSTEPS,
        stable_seed(SEED, "approx-production", interval_index),
    )
    reference_bank = generate_bridge_bank(
        dt, params["volatilities"], params["correlation"], REFERENCE_INNER_PATHS, REFERENCE_SUBSTEPS,
        stable_seed(SEED, "approx-reference", interval_index),
    )
    production = segment_probabilities(x, b, dt, params["volatilities"], params["correlation"], production_bank)
    reference = segment_probabilities(x, b, dt, params["volatilities"], params["correlation"], reference_bank)
    for local_index, path_index in enumerate(sample_index):
        p_ref_direct = reference["p3_nested_direct"][local_index]
        q_ref_direct = reference["q3_nested_direct"][local_index]
        audit_rows.append({
            "interval_index": interval_index + 1,
            "observation_time": contract.observation_times[interval_index],
            "path_index": int(path_index),
            "production_p3": production["p3"][local_index],
            "reference_p3": reference["p3"][local_index],
            "p3_error": production["p3"][local_index] - reference["p3"][local_index],
            "production_nested_direct_p3": production["p3_nested_direct"][local_index],
            "reference_nested_direct_p3": reference["p3_nested_direct"][local_index],
            "nested_direct_p3_error": (
                production["p3_nested_direct"][local_index]
                - reference["p3_nested_direct"][local_index]
            ),
            "production_q3": production["q3"][local_index],
            "reference_q3": reference["q3"][local_index],
            "q3_error": production["q3"][local_index] - reference["q3"][local_index],
            "production_q3_controlled_raw": production["q3_controlled_raw"][local_index],
            "reference_q3_controlled_raw": reference["q3_controlled_raw"][local_index],
            "q3_controlled_raw_error": (
                production["q3_controlled_raw"][local_index]
                - reference["q3_controlled_raw"][local_index]
            ),
            "production_q3_nested_direct": production["q3_nested_direct"][local_index],
            "reference_q3_nested_direct": reference["q3_nested_direct"][local_index],
            "q3_nested_direct_error": (
                production["q3_nested_direct"][local_index]
                - reference["q3_nested_direct"][local_index]
            ),
            "production_projection_abs": abs(
                production["q3"][local_index] - production["q3_controlled_raw"][local_index]
            ),
            "reference_bernoulli_se_proxy": math.sqrt(
                max(q_ref_direct * (1.0 - q_ref_direct), 0.0) / REFERENCE_INNER_PATHS
            ),
        })
approximation_bias_detail = pd.DataFrame(audit_rows)
approximation_bias_summary = (
    approximation_bias_detail.groupby("interval_index", as_index=False)
    .agg(
        observation_time=("observation_time", "first"),
        sample_endpoints=("path_index", "count"),
        mean_bias=("q3_controlled_raw_error", "mean"),
        mae=("q3_controlled_raw_error", lambda values: np.mean(np.abs(values))),
        rmse=("q3_controlled_raw_error", lambda values: math.sqrt(np.mean(np.square(values)))),
        max_abs_error=("q3_controlled_raw_error", lambda values: np.max(np.abs(values))),
        projected_q3_rmse=("q3_error", lambda values: math.sqrt(np.mean(np.square(values)))),
        nested_direct_q3_rmse=("q3_nested_direct_error", lambda values: math.sqrt(np.mean(np.square(values)))),
        auxiliary_direct_survival_rmse=("nested_direct_p3_error", lambda values: math.sqrt(np.mean(np.square(values)))),
        projected_formula_rmse=("p3_error", lambda values: math.sqrt(np.mean(np.square(values)))),
        mean_reference_se_proxy=("reference_bernoulli_se_proxy", "mean"),
        mean_production_projection=("production_projection_abs", "mean"),
    )
)
display(approximation_bias_summary)

## 6. Independence, permutation, probability-bound and underflow diagnostics

In [ ]:
rng = np.random.default_rng(stable_seed(SEED, "invariance"))
x = rng.normal(0.0, 0.08, size=(128, 3))
b = rng.uniform(0.12, 0.30, size=(128, 3))
sigma = np.array([0.18, 0.23, 0.29])
independent = segment_probabilities(x, b, 0.5, sigma, np.eye(3), bank=None)
independence_error = np.max(np.abs(independent["p3"] - np.prod(1.0 - independent["g"], axis=1)))

corr = np.array([[1.0, 0.35, 0.20], [0.35, 1.0, 0.45], [0.20, 0.45, 1.0]])
bank = generate_bridge_bank(0.5, sigma, corr, 2048, 32, stable_seed(SEED, "permutation-bank"))
base = segment_probabilities(x, b, 0.5, sigma, corr, bank)
perm = np.array([2, 0, 1])
permuted_bank = BridgeBank(
    residuals=bank.residuals[:, :, perm],
    time_fractions=bank.time_fractions,
    dt=bank.dt,
    inner_paths=bank.inner_paths,
    substeps=bank.substeps,
    seed=bank.seed,
)
permuted = segment_probabilities(
    x[:, perm], b[:, perm], 0.5, sigma[perm], corr[np.ix_(perm, perm)], permuted_bank
)
permutation_error = np.max(np.abs(base["p3"] - permuted["p3"]))

invariance_checks = pd.DataFrame([
    {"check": "correlation_to_zero_degeneration", "max_abs_error": independence_error, "tolerance": 1e-12, "pass": independence_error <= 1e-12},
    {"check": "symmetric_asset_permutation", "max_abs_error": permutation_error, "tolerance": 1e-11, "pass": permutation_error <= 1e-11},
])
probability_bounds_summary = (
    segment_diagnostics.groupby(["contract_id", "method"], as_index=False)
    .agg(
        raw_projection_rate=("raw_probability_bound_violation_rate", "mean"),
        mean_projection_abs=("q3_projection_abs_mean", "mean"),
        max_projection_abs=("q3_projection_abs_max", "max"),
        max_projection_p99=("q3_projection_abs_p99", "max"),
        final_bound_violation_rate=("final_probability_bound_violation_rate", "max"),
        numerical_underflow_count=("numerical_underflow_count", "sum"),
        endpoint_breach_rate=("endpoint_breach_rate", "mean"),
    )
)
runtime_by_component = (
    segment_diagnostics.groupby(["contract_id", "method"], as_index=False)
    .agg(
        marginal_g_seconds=("runtime_g_seconds", "sum"),
        bivariate_h_seconds=("runtime_h_seconds", "sum"),
        trivariate_q3_seconds=("runtime_q3_seconds", "sum"),
    )
)
display(invariance_checks)
display(probability_bounds_summary)
display(runtime_by_component)

## 7. Gate decision

In [ ]:
unit_test = subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", str(PROJECT_ROOT / "tests"), "-p", "test_autocallable_bb.py", "-q"],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
)
max_identity = replications["component_identity_error"].abs().max()
max_mass = replications["probability_mass_error"].abs().max()
max_root = replications["fair_coupon_residual"].abs().max()
max_approx_rmse = approximation_bias_summary["rmse"].max()
max_approx_error = approximation_bias_summary["max_abs_error"].max()
max_projection_mean = probability_bounds_summary["mean_projection_abs"].max()
max_projection_abs = probability_bounds_summary["max_projection_abs"].max()
max_projection_p99 = probability_bounds_summary["max_projection_p99"].max()

gate_rows = [
    {"gate": "Day 6 unit tests", "metric": f"returncode={unit_test.returncode}", "threshold": "0", "pass": unit_test.returncode == 0},
    {"gate": "M0-M3 large-budget price agreement", "metric": f"max_abs_diff={price_agreement.price_difference.abs().max():.6f}", "threshold": "scenario tolerance >= 1.25/100", "pass": bool(price_agreement["pass"].all())},
    {"gate": "paired Bernoulli bridge vs conditional weight", "metric": f"max_abs_mean_diff={paired_comparison.paired_mean_difference.abs().max():.6f}", "threshold": "max(0.35, 3 SE)", "pass": bool(paired_comparison["pass"].all())},
    {"gate": "correlation to zero and permutation invariance", "metric": f"max_error={invariance_checks.max_abs_error.max():.3e}", "threshold": "individual tolerances", "pass": bool(invariance_checks["pass"].all())},
    {"gate": "probability bounds and numerical underflow", "metric": f"final_violation={probability_bounds_summary.final_bound_violation_rate.max():.3e}; underflow={probability_bounds_summary.numerical_underflow_count.sum()}", "threshold": "0; projection explicitly audited", "pass": bool((probability_bounds_summary.final_bound_violation_rate == 0).all() and probability_bounds_summary.numerical_underflow_count.sum() == 0)},
    {"gate": "fresh nested approximation error", "metric": f"max_RMSE={max_approx_rmse:.6f}; max_abs={max_approx_error:.6f}", "threshold": "RMSE <= 0.03; max abs <= 0.10", "pass": bool(max_approx_rmse <= 0.03 and max_approx_error <= 0.10)},
    {"gate": "q3 feasible-projection magnitude", "metric": f"max_mean={max_projection_mean:.6f}; max_p99={max_projection_p99:.6f}; observed_max={max_projection_abs:.6f}", "threshold": "mean <= 0.005; p99 <= 0.02; maximum retained as warning", "pass": bool(max_projection_mean <= 0.005 and max_projection_p99 <= 0.02)},
    {"gate": "cash-flow identities and fair-coupon roots", "metric": f"identity={max_identity:.3e}; mass={max_mass:.3e}; root={max_root:.3e}", "threshold": "all <= 1e-10", "pass": bool(max(max_identity, max_mass, max_root) <= 1e-10)},
]
gate_summary = pd.DataFrame(gate_rows)
DAY6_PASS = bool(gate_summary["pass"].all())
display(gate_summary)
print("DAY 6 STATUS:", "PASS" if DAY6_PASS else "FAIL — do not enter Greeks")
if not DAY6_PASS:
    print("Greeks remain blocked until every price/probability gate passes.")

## 8. Evidence charts and exported audit tables

In [ ]:
palette = {"M0": "#2F6B9A", "M1": "#B88900", "M2": "#D97706", "M3": "#6B7F3A"}
markers = {"M0": "o", "M1": "s", "M2": "^", "M3": "D"}
fig, axes = plt.subplots(1, 2, figsize=(12.6, 4.8), constrained_layout=True)
for ax, contract_id in zip(axes, ("RC-L", "RC-A")):
    subset = method_summary[method_summary.contract_id == contract_id].set_index("method").loc[["M0", "M1", "M2", "M3"]]
    positions = np.arange(4)
    for position, (method, row) in enumerate(subset.iterrows()):
        ax.errorbar(
            position, row.total_value_mean, yerr=1.96 * row.total_value_se,
            fmt=markers[method], color=palette[method], markerfacecolor="white",
            markeredgewidth=1.6, capsize=4, linewidth=1.5, markersize=7,
        )
    ax.set_xticks(positions, subset.index)
    ax.set_title(contract_id, loc="left", pad=12)
    ax.set_ylabel("Present value per 100 notional")
    ax.grid(axis="y")
    ax.text(
        0.0, 1.015, "Dots: replication means; bars: 95% outer-replication CI",
        transform=ax.transAxes, fontsize=8.5, color="#64748b",
    )
fig.suptitle("M0-M3 price agreement", x=0.01, ha="left", fontsize=14, fontweight="semibold")
fig.savefig(OUTPUT_DIR / "m0_m3_price_agreement.png", bbox_inches="tight", facecolor="white")
plt.show()

m3_segment = (
    segment_diagnostics[segment_diagnostics.method == "M3"]
    .groupby(["contract_id", "observation_index"], as_index=False)
    .agg(observation_time=("observation_time", "first"), p3_mean=("p3_mean", "mean"))
)
m3_weights = (
    weight_distribution[(weight_distribution.method == "M3") & (weight_distribution["quantile"] == 0.5)]
    .groupby(["contract_id", "observation_index"], as_index=False)
    .agg(observation_time=("observation_time", "first"), median_weight=("cumulative_survival_weight", "mean"))
)
fig, axes = plt.subplots(1, 2, figsize=(12.6, 4.8), constrained_layout=True)
for contract_id, color, marker, linestyle in (("RC-L", "#2F6B9A", "o", "-"), ("RC-A", "#D97706", "s", "--")):
    p = m3_segment[m3_segment.contract_id == contract_id]
    w = m3_weights[m3_weights.contract_id == contract_id]
    axes[0].plot(p.observation_index, p.p3_mean, marker=marker, linestyle=linestyle, color=color, markerfacecolor="white", linewidth=2.0, markersize=6.5, label=contract_id)
    axes[1].plot(w.observation_index, w.median_weight, marker=marker, linestyle=linestyle, color=color, markerfacecolor="white", linewidth=2.0, markersize=6.5, label=contract_id)
axes[0].set_title("Mean segment survival probability", loc="left", pad=12)
axes[1].set_title("Median cumulative survival weight", loc="left", pad=12)
subtitles = ("Endpoint-conditioned p3 across live paths", "Weights stop evolving after early redemption")
for ax, subtitle in zip(axes, subtitles):
    ax.set_xlabel("Observation index")
    ax.set_ylabel("Probability / weight")
    ax.set_ylim(-0.02, 1.02)
    ax.grid(True)
    ax.legend(ncol=2)
    ax.text(0.0, 1.015, subtitle, transform=ax.transAxes, fontsize=8.5, color="#64748b")
fig.suptitle("How M3 continuous-KI weights evolve", x=0.01, ha="left", fontsize=14, fontweight="semibold")
fig.savefig(OUTPUT_DIR / "m3_segment_survival_weights.png", bbox_inches="tight", facecolor="white")
plt.show()

exports = {
    "m0_m3_replications.csv": replications,
    "m0_m3_summary.csv": method_summary,
    "price_agreement.csv": price_agreement,
    "component_comparison.csv": component_comparison,
    "paired_bernoulli_comparison.csv": paired_comparison,
    "segment_probability_diagnostics.csv": segment_diagnostics,
    "weight_distribution.csv": weight_distribution,
    "approximation_bias_detail.csv": approximation_bias_detail,
    "approximation_bias_summary.csv": approximation_bias_summary,
    "invariance_checks.csv": invariance_checks,
    "probability_bounds_summary.csv": probability_bounds_summary,
    "runtime_by_probability_component.csv": runtime_by_component,
    "gate_summary.csv": gate_summary,
    "appendix_probability_check.csv": appendix_check,
}
for filename, frame in exports.items():
    frame.to_csv(OUTPUT_DIR / filename, index=False)

manifest = pd.DataFrame([
    {"field": "status", "value": "PASS" if DAY6_PASS else "FAIL"},
    {"field": "direct_paths", "value": DIRECT_PATHS},
    {"field": "direct_replications", "value": DIRECT_REPLICATIONS},
    {"field": "direct_steps_per_year", "value": DIRECT_STEPS_PER_YEAR},
    {"field": "conditioned_paths", "value": CONDITIONED_PATHS},
    {"field": "conditioned_replications", "value": CONDITIONED_REPLICATIONS},
    {"field": "inner_paths", "value": INNER_PATHS},
    {"field": "bridge_substeps", "value": BRIDGE_SUBSTEPS},
    {"field": "reference_inner_paths", "value": REFERENCE_INNER_PATHS},
    {"field": "reference_substeps", "value": REFERENCE_SUBSTEPS},
    {"field": "bessel_terms", "value": BESSEL_TERMS},
    {"field": "day4_logistic_reused", "value": False},
    {"field": "fallback", "value": "fresh scrambled-Sobol nested conditional bridge bank; median of three exact-h anchored conditional ratios"},
    {"field": "source_paper", "value": "Lee et al. (2024), N.A.J.E.F. 73, 102174"},
])
manifest.to_csv(OUTPUT_DIR / "run_manifest.csv", index=False)
print("Saved", len(exports) + 1, "CSV evidence files and 2 PNG figures to", OUTPUT_DIR)

## Conclusion

Day 6 passes only if every row in `gate_summary.csv` passes. The notebook explicitly
separates:

1. **pricing sampling error** — replication SD/SE for M0–M3;
2. **nested approximation error** — production-vs-high-budget interval (p_3) audit;
3. **feasible-projection diagnostics** — frequency and magnitude of (q_3) projection;
4. **paired bridge noise** — conditional weights versus paired Bernoulli bridge draws.

If the price gate fails, the project does not proceed to Greeks. A PASS authorises the
planned Day 7 estimator-comparison work; it does not retroactively validate the Day 4 logistic.